# SpaceX Falcon 9 First-Stage Landing Prediction

## Lab 1: Collecting the Data

This notebook collects historical launch data from the SpaceX API, enriches ID fields with human-readable information, filters the data to Falcon 9 launches, handles missing payload masses, and exports `dataset_part_1.csv` for the next capstone lab.

> Run the notebook from top to bottom in an environment with internet access. The course uses a static launch-response file so that results remain consistent across learners.


## Objectives

- Request and parse SpaceX launch data.
- Retrieve booster, payload, launch-site, and core details.
- Construct a clean launch-level DataFrame.
- Keep Falcon 9 launches only.
- Replace missing payload masses with the column mean.
- Export the cleaned dataset.


In [ ]:
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


## Helper functions


In [ ]:
def get_booster_version(data: pd.DataFrame) -> list:
    """Return the booster name associated with each rocket ID."""
    booster_versions = []
    for rocket_id in data["rocket"]:
        response = requests.get(
            f"https://api.spacexdata.com/v4/rockets/{rocket_id}",
            timeout=30,
        )
        response.raise_for_status()
        booster_versions.append(response.json()["name"])
    return booster_versions


def get_launch_site(data: pd.DataFrame) -> tuple[list, list, list]:
    """Return launch-site names, longitudes, and latitudes."""
    launch_sites, longitudes, latitudes = [], [], []
    for launchpad_id in data["launchpad"]:
        response = requests.get(
            f"https://api.spacexdata.com/v4/launchpads/{launchpad_id}",
            timeout=30,
        )
        response.raise_for_status()
        launchpad = response.json()
        launch_sites.append(launchpad["name"])
        longitudes.append(launchpad["longitude"])
        latitudes.append(launchpad["latitude"])
    return launch_sites, longitudes, latitudes


def get_payload_data(data: pd.DataFrame) -> tuple[list, list]:
    """Return payload mass and target orbit for each payload ID."""
    payload_masses, orbits = [], []
    for payload_id in data["payloads"]:
        response = requests.get(
            f"https://api.spacexdata.com/v4/payloads/{payload_id}",
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        payload_masses.append(payload["mass_kg"])
        orbits.append(payload["orbit"])
    return payload_masses, orbits


def get_core_data(data: pd.DataFrame) -> dict[str, list]:
    """Return core and landing information for each launch."""
    core_data = {
        "Outcome": [],
        "Flights": [],
        "GridFins": [],
        "Reused": [],
        "Legs": [],
        "LandingPad": [],
        "Block": [],
        "ReusedCount": [],
        "Serial": [],
    }

    for core in data["cores"]:
        if core["core"] is not None:
            response = requests.get(
                f"https://api.spacexdata.com/v4/cores/{core['core']}",
                timeout=30,
            )
            response.raise_for_status()
            core_response = response.json()
            core_data["Block"].append(core_response["block"])
            core_data["ReusedCount"].append(core_response["reuse_count"])
            core_data["Serial"].append(core_response["serial"])
        else:
            core_data["Block"].append(None)
            core_data["ReusedCount"].append(None)
            core_data["Serial"].append(None)

        core_data["Outcome"].append(
            f"{core['landing_success']} {core['landing_type']}"
        )
        core_data["Flights"].append(core["flight"])
        core_data["GridFins"].append(core["gridfins"])
        core_data["Reused"].append(core["reused"])
        core_data["Legs"].append(core["legs"])
        core_data["LandingPad"].append(core["landpad"])

    return core_data


## Task 1: Request and parse the SpaceX launch data


In [ ]:
# Optional live endpoint: useful for inspecting the current API response.
spacex_url = "https://api.spacexdata.com/v4/launches/past"
live_response = requests.get(spacex_url, timeout=30)
live_response.raise_for_status()

print("Live API status code:", live_response.status_code)
print("Number of launches returned by live endpoint:", len(live_response.json()))


In [ ]:
# The course uses this static response to keep results consistent.
static_json_url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
)

response = requests.get(static_json_url, timeout=30)
response.raise_for_status()
print("Static dataset status code:", response.status_code)


In [ ]:
data = pd.json_normalize(response.json())
data.head()


The raw launch response contains IDs for rockets, payloads, launchpads, and cores.  
The following cells retain the needed fields and use the IDs to request descriptive information.


In [ ]:
data = data[
    ["rocket", "payloads", "launchpad", "cores", "flight_number", "date_utc"]
].copy()

# Exclude launches with multiple cores or payloads so each row maps to one core and payload.
data = data[data["cores"].map(len) == 1]
data = data[data["payloads"].map(len) == 1]

# Extract the single item from each one-element list.
data["cores"] = data["cores"].map(lambda value: value[0])
data["payloads"] = data["payloads"].map(lambda value: value[0])

# Convert the timestamp and restrict the dataset to the course cut-off date.
data["date"] = pd.to_datetime(data["date_utc"]).dt.date
data = data[data["date"] <= datetime.date(2020, 11, 13)].copy()

data.head()


In [ ]:
booster_versions = get_booster_version(data)
launch_sites, longitudes, latitudes = get_launch_site(data)
payload_masses, orbits = get_payload_data(data)
core_details = get_core_data(data)

launch_dict = {
    "FlightNumber": data["flight_number"].tolist(),
    "Date": data["date"].tolist(),
    "BoosterVersion": booster_versions,
    "PayloadMass": payload_masses,
    "Orbit": orbits,
    "LaunchSite": launch_sites,
    "Outcome": core_details["Outcome"],
    "Flights": core_details["Flights"],
    "GridFins": core_details["GridFins"],
    "Reused": core_details["Reused"],
    "Legs": core_details["Legs"],
    "LandingPad": core_details["LandingPad"],
    "Block": core_details["Block"],
    "ReusedCount": core_details["ReusedCount"],
    "Serial": core_details["Serial"],
    "Longitude": longitudes,
    "Latitude": latitudes,
}

launch_data = pd.DataFrame(launch_dict)
launch_data.head()


In [ ]:
print("Dataset shape before Falcon 9 filtering:", launch_data.shape)
launch_data.info()


## Task 2: Keep Falcon 9 launches only


In [ ]:
data_falcon9 = launch_data[
    launch_data["BoosterVersion"] != "Falcon 1"
].copy()

data_falcon9.reset_index(drop=True, inplace=True)
data_falcon9["FlightNumber"] = range(1, len(data_falcon9) + 1)

print("Falcon 9 dataset shape:", data_falcon9.shape)
data_falcon9.head()


## Task 3: Handle missing values


In [ ]:
missing_before = data_falcon9.isna().sum()
missing_before


In [ ]:
payload_mass_mean = data_falcon9["PayloadMass"].mean()
print(f"Mean PayloadMass: {payload_mass_mean:.2f} kg")

data_falcon9["PayloadMass"] = data_falcon9["PayloadMass"].fillna(
    payload_mass_mean
)


In [ ]:
missing_after = data_falcon9.isna().sum()
missing_after


In [ ]:
# LandingPad may legitimately remain missing when no landing pad was used.
unexpected_missing = missing_after.drop(labels=["LandingPad"])
assert unexpected_missing.sum() == 0, (
    "Unexpected missing values remain:\n"
    f"{unexpected_missing[unexpected_missing > 0]}"
)

assert data_falcon9["BoosterVersion"].eq("Falcon 1").sum() == 0
assert data_falcon9["FlightNumber"].tolist() == list(
    range(1, len(data_falcon9) + 1)
)

print("Validation checks passed.")


## Export the cleaned dataset


In [ ]:
output_path = Path("dataset_part_1.csv")
data_falcon9.to_csv(output_path, index=False)

print(f"Saved {len(data_falcon9)} rows to {output_path.resolve()}")
data_falcon9.tail()


## Completion checklist

- All code cells run without errors.
- `data_falcon9` contains Falcon 9 launches only.
- `PayloadMass` has no missing values.
- Missing `LandingPad` values are retained because some launches did not use a landing pad.
- `dataset_part_1.csv` appears in the notebook's working directory.
- The notebook is saved with its outputs before it is uploaded to GitHub or submitted.
